# Saturation analysis: GO:BP ORA computation

**Environment:** `clamp-analyses`

Runs GO:BP ORA (enrichGO, top 1% genes per LV, `pvalueCutoff=1, qvalueCutoff=1`, `minGSSize=10, maxGSSize=500`) for each model in the saturation grid and caches results to `output/03_model_biology/00_archs4/00_pathway_coverage/07_saturation_gobp_ora/`.

Results are read by `07_saturation_coverage_gobp_plot.ipynb` to produce saturation curves.

## Load libraries

In [1]:
library(here)
library(dplyr)
library(clusterProfiler)
library(org.Hs.eg.db)
library(BiocParallel)

source(here("config.R"))

base_output_dir <- config$ARCHS4$DATASET_FOLDER

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




clusterProfiler v4.14.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

G Yu. Thirteen years of clusterProfiler. The Innovation. 2024,
5(6):100722


Attaching package: ‘clusterProfiler’


The following object is masked from ‘package:stats’:

    filter


Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, ba

## Define model grid

In [2]:
k_fracs <- c(0.05, 0.10, 0.25, 0.50, 0.75, 1.00)

pct_to_subdir <- c(
  "1"   = "00_bp_coverage_hall_rs_01",
  "5"   = "01_bp_coverage_hall_rs_05",
  "10"  = "02_bp_coverage_hall_rs_10",
  "25"  = "03_bp_coverage_hall_rs_25",
  "50"  = "04_bp_coverage_hall_rs_50",
  "75"  = "05_bp_coverage_hall_rs_75",
  "100" = "06_bp_coverage_hall_rs_100"
)

get_06_model_dir <- function(pct, seed_idx) {
  file.path(base_output_dir, "06_bp_coverage_rshall",
            pct_to_subdir[as.character(pct)],
            sprintf("hall_coverage_rs%d_seed_%d", pct, seed_idx))
}

K_100    <- readRDS(file.path(get_06_model_dir(100L, 1L), "CLAMP_K.rds"))
k_values <- sort(unique(as.integer(round(k_fracs * K_100))))
message("K_100 = ", K_100, " | k_values = ", paste(k_values, collapse = ", "))

data_coverages <- c(1L, 5L, 10L, 25L)  # expand to c(1L,5L,10L,25L,50L,75L,100L) as jobs complete
seed_indices   <- 1:3

model_grid <- expand.grid(data_pct = data_coverages, k_val = k_values,
                           seed_idx = seed_indices, stringsAsFactors = FALSE)
message("Model grid: ", nrow(model_grid), " combinations")

get_z_path <- function(data_pct, k_val, seed_idx) {
  file.path(base_output_dir, "07_saturation",
            sprintf("hall_saturation_rs%d_k%d_seed_%d", data_pct, k_val, seed_idx),
            "CLAMPfull_hall", "Z.csv")
}

K_100 = 1728 | k_values = 86, 173, 432, 864, 1296, 1728

Model grid: 72 combinations



## ORA cache directory

In [3]:
ora_cache_dir <- here("output/03_model_biology/00_archs4/00_pathway_coverage/07_saturation_gobp_ora")
dir.create(ora_cache_dir, recursive = TRUE, showWarnings = FALSE)
message("ORA cache dir: ", ora_cache_dir)

ORA cache dir: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/00_pathway_coverage/07_saturation_gobp_ora



## Helper: run GO:BP ORA for one saturation model

In [4]:
run_ora_saturation <- function(data_pct, k_val, seed_idx, n_cores = 16) {
  z_path <- get_z_path(data_pct, k_val, seed_idx)
  if (!file.exists(z_path)) return(NULL)

  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  n_top          <- ceiling(0.01 * nrow(Z))
  n_lvs          <- ncol(Z)
  lv_names       <- colnames(Z)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })

  bp <- BiocParallel::MulticoreParam(workers = n_cores, progressbar = FALSE)

  ora_list <- BiocParallel::bplapply(
    seq_len(n_lvs),
    function(i) {
      res <- tryCatch(
        clusterProfiler::enrichGO(
          gene          = top_genes_per_lv[, i],
          universe      = universe_genes,
          OrgDb         = org.Hs.eg.db,
          keyType       = "SYMBOL",
          ont           = "BP",
          pAdjustMethod = "BH",
          pvalueCutoff  = 1,
          qvalueCutoff  = 1,
          minGSSize     = 10,
          maxGSSize     = 500
        ),
        error = function(e) NULL
      )
      if (is.null(res) || nrow(as.data.frame(res)) == 0) return(NULL)
      df      <- as.data.frame(res)
      df$LV   <- lv_names[i]
      df
    },
    BPPARAM = bp
  )

  ora_df <- base::do.call(base::rbind, base::Filter(Negate(is.null), ora_list))
  if (is.null(ora_df) || nrow(ora_df) == 0) return(NULL)

  list(
    ora_df   = ora_df,
    data_pct = data_pct,
    k_val    = k_val,
    seed_idx = seed_idx,
    n_lvs    = n_lvs
  )
}

## Run ORA for all models (with caching)

In [5]:
for (i in seq_len(nrow(model_grid))) {
  data_pct <- model_grid$data_pct[i]
  k_val    <- model_grid$k_val[i]
  seed_idx <- model_grid$seed_idx[i]

  cache_path <- file.path(ora_cache_dir,
    sprintf("pct%d_k%d_seed%d.rds", data_pct, k_val, seed_idx))

  if (file.exists(cache_path)) {
    message(sprintf("Cached: pct%d k%d seed%d", data_pct, k_val, seed_idx))
    next
  }

  z_path <- get_z_path(data_pct, k_val, seed_idx)
  if (!file.exists(z_path)) {
    message(sprintf("Not found: pct%d k%d seed%d", data_pct, k_val, seed_idx))
    next
  }

  message(sprintf("Running ORA: pct%d k%d seed%d", data_pct, k_val, seed_idx))
  result <- run_ora_saturation(data_pct, k_val, seed_idx)

  if (!is.null(result)) {
    saveRDS(result, cache_path)
    message(sprintf("  Saved: %d terms, %d LVs",
                    length(unique(result$ora_df$ID)), result$n_lvs))
  } else {
    message(sprintf("  No ORA results for pct%d k%d seed%d", data_pct, k_val, seed_idx))
  }
}

message("ORA complete.")

Cached: pct1 k86 seed1

Cached: pct5 k86 seed1

Cached: pct10 k86 seed1

Cached: pct25 k86 seed1

Cached: pct1 k173 seed1

Cached: pct5 k173 seed1

Cached: pct10 k173 seed1

Cached: pct25 k173 seed1

Cached: pct1 k432 seed1

Cached: pct5 k432 seed1

Cached: pct10 k432 seed1

Cached: pct25 k432 seed1

Running ORA: pct1 k864 seed1

  Saved: 5879 terms, 864 LVs

Running ORA: pct5 k864 seed1

  Saved: 5879 terms, 864 LVs

Running ORA: pct10 k864 seed1

  Saved: 5879 terms, 864 LVs

Running ORA: pct25 k864 seed1

  Saved: 5879 terms, 864 LVs

Running ORA: pct1 k1296 seed1

  Saved: 5879 terms, 1296 LVs

Running ORA: pct5 k1296 seed1

  Saved: 5879 terms, 1296 LVs

Running ORA: pct10 k1296 seed1

  Saved: 5879 terms, 1296 LVs

Not found: pct25 k1296 seed1

Not found: pct1 k1728 seed1

Running ORA: pct5 k1728 seed1

  Saved: 5879 terms, 1728 LVs

Running ORA: pct10 k1728 seed1

  Saved: 5879 terms, 1728 LVs

Not found: pct25 k1728 seed1

Not found: pct1 k86 seed2

Running ORA: pct5 k86 seed2
